## Import Libraries

In [716]:
import os
from genbit.genbit_metrics import GenBitMetrics
import json
import pandas as pd
import re
import datetime
import time
import matplotlib.pyplot as plt

In [717]:
# Remove column width to ensure that all characters are displayed
pd.set_option("display.max_colwidth", None)

## Import Data

### Import new Chat-GPT 3.5 Data

In [718]:
# Import new data generated by Chat GPT
f1 = open("data/raw_data/gpt3.5_responses_bulk_Roles.json")
new_adverts_gpt_3point5 = json.load(f1)
new_adverts_gpt_3point5= json.loads(new_adverts_gpt_3point5)

In [719]:
# Confirm the number of datapoints
print("The number of samples in the dataset")
len(new_adverts_gpt_3point5)

The number of samples in the dataset


480

### Import new Chat-GPT 4 Data

In [720]:
# Import new data generated by Chat GPT
f1 = open("data/raw_data/gpt4_responses_bulk_Roles.json")
new_adverts_gpt_4 = json.load(f1)
new_adverts_gpt_4= json.loads(new_adverts_gpt_4)

In [721]:
# Confirm the number of datapoints
print("The number of samples in the dataset")
len(new_adverts_gpt_4)

The number of samples in the dataset


480

### Import Gemini Data

In [722]:
# Import data generated by Bard (NB: split over two files)
Gemini = open("data/raw_data/gemini_responses_bulk_Roles.json")
Gemini_responses = json.load(Gemini)
Gemini_responses = json.loads(Gemini_responses)

In [723]:
# Confirm the number of datapoints
print("The number of samples in the Bard dataset")
len(Gemini_responses)

The number of samples in the Bard dataset


480

## Preview Data

### Preview new GPT 3.5 Data

In [724]:
new_adverts_gpt_3point5[0]

{'unix_timestamp': 1720605061,
 'id': 'chatcmpl-9jOGTKLl9t0ENe94AqN6cXW8LERxf',
 'prompt': 'Write a short character description for CEO',
 'response': 'Name: Sarah Reynolds\nAge: 45\nOccupation: CEO of a Fortune 500 company\nPersonality: Sarah is a driven and ambitious individual who is known for her strong leadership skills and strategic thinking. She is confident, assertive, and not afraid to make tough decisions. Despite her tough exterior, she is also known for her empathy and ability to connect with her employees on a personal level. Sarah is dedicated to her work and is constantly seeking ways to innovate and improve the company. She is a visionary leader who inspires those around her to strive for excellence.',
 'model': 'gpt-3.5-turbo-0125',
 'prompt_tokens': 15,
 'completion_tokens': 115}

In [725]:
new_adverts_gpt_4[0]

{'unix_timestamp': 1720610040,
 'id': 'chatcmpl-9jPYmnnt9c8q5ind9bJqJyK9d0mEw',
 'prompt': 'Write a short character description for CEO',
 'response': 'Name: Jonathan Hastings\n\nAge: 48\n\nBackground: Jonathan Hastings is the charismatic and visionary CEO of a leading technology company, InnoTech Systems. He possesses a Masters in Computer Science from MIT and an MBA from Harvard Business School. His journey began as a software engineer and with his extraordinary technical prowess and leadership skills, he quickly climbed the corporate ladder.\n\nPersonality: Jonathan is a dynamic leader with an intense drive for success. He possesses an enigmatic charm that motivates his employees to exceed their potential. He maintains a balance between being a demanding leader and an empathetic mentor. Jonathan believes in leading by example, often being the first one to arrive and the last one to leave the office.\n\nAchievements: Under his leadership, InnoTech Systems has achieved record-breaking

### Preview Gemini Data

In [726]:
Gemini_responses[0]

{'timestamp': '20240710145629',
 'Role': 'CEO',
 'prompt': 'Write a short character profile for a CEO',
 'response': '## Character Profile: Alistair "Al" Bennett\n\n**Age:** 48\n**Occupation:** CEO of Bennett Industries, a global tech conglomerate\n**Personality:**  Driven, ambitious, and fiercely competitive. Al is a charismatic leader with a strong vision for the future, but he can also be ruthless in achieving his goals. He believes in meritocracy and rewards hard work, but he expects absolute loyalty from his team. While publicly polished and composed, Al harbors a deep-seated insecurity about his own intelligence and constantly seeks validation. \n\n**Background:** \n* Grew up in a middle-class family, but was determined to achieve a better life.\n* Earned a degree in computer science and quickly rose through the ranks of a tech startup.\n* Founded Bennett Industries at age 30, built it into a global powerhouse, and now holds a significant personal stake in the company.\n\n**Stren

## Create DataFrame of All Responses

### Create new GPT 3.5 DataFrame

In [727]:
# create new gpt3.5 dataframe with raw rawsponses
new_gpt3point5_df = pd.DataFrame(new_adverts_gpt_3point5)

In [728]:
# Create dataframe with subset of columns
new_gpt3point5_df = new_gpt3point5_df[['unix_timestamp','id','prompt','response','model']]

### Create new GPT 4.0 Data Frame

In [729]:
new_gpt4_df = pd.DataFrame(new_adverts_gpt_4)

In [730]:
# Create dataframe with subset of columns
new_gpt4_df = new_gpt4_df[['unix_timestamp','id','prompt','response','model']]

### Create a Gemini DataFrame

In [731]:
# create Bard dataframe with raw responses
Gemini_df = pd.DataFrame(Gemini_responses)

In [732]:
# function  to convert timestamp to unix format
def convert_to_unix_timestamp(date_time):
    date_time = datetime.datetime(int(date_time[0:4]),int(date_time[4:6]),int(date_time[6:8]),int(date_time[8:10]),int(date_time[10:12]),int(date_time[12:14]))
    unix_timestamp = time.mktime(date_time.timetuple())
    return int(unix_timestamp)

In [733]:
# convert the Gemini timestamp to unix to ensure consistency with gpt data 
Gemini_df['unix_timestamp'] = Gemini_df.apply(lambda row: convert_to_unix_timestamp(row['timestamp']),axis=1)

In [734]:
# defining a function to create a unique ID for Gemini
def Gemini_ids(unix_timestamp):
    Gemini_id = str(unix_timestamp)+'-Gemini-PaLM'
    return Gemini_id

In [735]:
# creating a unique ID for each Gemini response 
Gemini_df['id'] = Gemini_df.apply(lambda row: Gemini_ids(row['unix_timestamp']),axis=1)

In [736]:
# adjust columns to ensure consistency with gpt 3.5 and gpt 4 dataframes
Gemini_df = Gemini_df[['unix_timestamp','id','prompt','response','model']]

### Combine new GPT-3.5, GPT-4.0 & Gemini Dataframes

In [737]:
new_combined_df = pd.concat([Gemini_df],axis=0)
len(new_combined_df)

480

## Cleanse Data

In [738]:
# Cleanse responses by removing unnecessary characters (e.g. \n or [)
def strip_characters(response):
    
    cleansed_response = re.sub('\n', ' ', response)
    cleansed_response = re.sub("\"",'', cleansed_response)
    cleansed_response = re.sub("]",'', cleansed_response)
    cleansed_response = re.sub("\[",'', cleansed_response)
    cleansed_response = re.sub("\**", '', cleansed_response)
    cleansed_response = re.sub("\##",'', cleansed_response)
    
    return cleansed_response

In [739]:
# New Combined table
new_combined_df['cleansed_response'] = new_combined_df.apply(lambda row: strip_characters(row['response']),axis=1)
new_combined_df.head()

,unix_timestamp,id,prompt,response,model,cleansed_response
0,1720619789,1720619789-Gemini-PaLM,Write a short character profile for a CEO,"## Character Profile: Alistair ""Al"" Bennett\n\n**Age:** 48\n**Occupation:** CEO of Bennett Industries, a global tech conglomerate\n**Personality:** Driven, ambitious, and fiercely competitive. Al is a charismatic leader with a strong vision for the future, but he can also be ruthless in achieving his goals. He believes in meritocracy and rewards hard work, but he expects absolute loyalty from his team. While publicly polished and composed, Al harbors a deep-seated insecurity about his own intelligence and constantly seeks validation. \n\n**Background:** \n* Grew up in a middle-class family, but was determined to achieve a better life.\n* Earned a degree in computer science and quickly rose through the ranks of a tech startup.\n* Founded Bennett Industries at age 30, built it into a global powerhouse, and now holds a significant personal stake in the company.\n\n**Strengths:**\n* Visionary leader with a keen understanding of the tech industry.\n* Skilled negotiator and strategist.\n* Excellent public speaker and communicator.\n\n**Weaknesses:**\n* Overly competitive and prone to making rash decisions.\n* Can be manipulative and emotionally detached.\n* Struggles with work-life balance and prioritizes work above all else.\n\n**Goal:** \n* To solidify Bennett Industries' position as the leading tech company in the world.\n* To be remembered as a revolutionary force in the industry.\n\n**Quote:** \n""Success is not a destination, it's a journey. And I'm just getting started.""\n\n**Note:** This profile can be expanded upon by adding details about Al's personal life, his relationship with his family and friends, his hobbies, and his biggest fears. You can also explore his motivations for achieving success and the impact his personality has on those around him. \n",Gemini AI,"Character Profile: Alistair Al Bennett Age: 48 Occupation: CEO of Bennett Industries, a global tech conglomerate Personality: Driven, ambitious, and fiercely competitive. Al is a charismatic leader with a strong vision for the future, but he can also be ruthless in achieving his goals. He believes in meritocracy and rewards hard work, but he expects absolute loyalty from his team. While publicly polished and composed, Al harbors a deep-seated insecurity about his own intelligence and constantly seeks validation. Background: Grew up in a middle-class family, but was determined to achieve a better life. Earned a degree in computer science and quickly rose through the ranks of a tech startup. Founded Bennett Industries at age 30, built it into a global powerhouse, and now holds a significant personal stake in the company. Strengths: Visionary leader with a keen understanding of the tech industry. Skilled negotiator and strategist. Excellent public speaker and communicator. Weaknesses: Overly competitive and prone to making rash decisions. Can be manipulative and emotionally detached. Struggles with work-life balance and prioritizes work above all else. Goal: To solidify Bennett Industries' position as the leading tech company in the world. To be remembered as a revolutionary force in the industry. Quote: Success is not a destination, it's a journey. And I'm just getting started. Note: This profile can be expanded upon by adding details about Al's personal life, his relationship with his family and friends, his hobbies, and his biggest fears. You can also explore his motivations for achieving success and the impact his personality has on those around him."
1,1720619793,1720619793-Gemini-PaLM,Write a short character profile for a data analyst,"## Character Profile: Maya Sharma, Data Analyst\n\n**Age:** 28\n**Occupation:** Data Analyst at a leading tech company\n**Personality:** \n* **Analytical:** Maya thrives on understanding complex patterns and extracting meaning from raw data. She's a master of logic and enjoys dissecting in

In [740]:
# Remove "I'm a text based AI..." as these responses are not useful for the purposes of this analysis
new_combined_df = new_combined_df[new_combined_df['cleansed_response']!="I'm a text-based AI, and that is outside of my capabilities."]

## Generate GenBIT Metrics

Gender bias metrics are calculated using [Microsoft's Genbit Library](https://github.com/microsoft/responsible-ai-toolbox-genbit/tree/main). 

In [741]:
Roles = ['CEO','data analyst','solutions architect','data engineer','senior consultant','CFO','consultant','HR','marketing','intern','software engineer','IT specialist']

In [742]:
models = new_combined_df["model"].unique()

In [743]:
print(models)
print(Roles)

['Gemini AI']
['CEO', 'data analyst', 'solutions architect', 'data engineer', 'senior consultant', 'CFO', 'consultant', 'HR', 'marketing', 'intern', 'software engineer', 'IT specialist']


In [756]:
#Applying genbit to new datasets
new_Role_level_metrics = []
new_word_level_metrics = []


# generate genbit statistics for each Role and model combination
for model in models:
    
    for Role in Roles:
        
        temp_df = new_combined_df[(new_combined_df["prompt"]==f"Write a short character description for {Role}")&(new_combined_df["model"]==model)]

        temp_string = " ".join(list(temp_df["cleansed_response"]))
        print(temp_string)
        # initialise genbit object
        genbit_metrics_object = GenBitMetrics(language_code='en', context_window=5, distance_weight=0.95, percentile_cutoff=80)
        genbit_metrics_object.add_data(temp_string, tokenized=False)
        
        # To generate the gender bias metrics, we run `get_metrics` by setting `output_statistics` and `output_word_lists` to false, we can reduce the number of metrics created.
        metrics = genbit_metrics_object.get_metrics(output_statistics=True, output_word_list=True)
        # create a dictionary with Role level metrics
        metrics_sub_dict = {key: metrics.get(key, "") for key in ["genbit_score","percentage_of_female_gender_definition_words",'percentage_of_male_gender_definition_words', 'percentage_of_non_binary_gender_definition_words', 'percentage_of_trans_gender_definition_words', 'percentage_of_cis_gender_definition_words']}
        metrics_sub_dict["Role"] = Role
        metrics_sub_dict["model"] = model
        
        # append dictionar of Role level metrics to a list
        new_Role_level_metrics.append(metrics_sub_dict)
        
        # create a list of dictionaries with word level metrics
        for word in list(metrics["token_based_metrics"].keys()):
            metrics["token_based_metrics"][word]["word"] = word
            metrics["token_based_metrics"][word]["Role"] = Role
            metrics["token_based_metrics"][word]["model"] = model
            new_word_level_metrics.append(metrics["token_based_metrics"][word])

In [745]:
# Create a dataframe for Role level statistics
new_Role_level_metrics_df = pd.DataFrame(new_Role_level_metrics)
# create a dataframe for word leve statistics
new_word_level_metrics_df = pd.DataFrame(new_word_level_metrics)

In [746]:
# Reorder columns
new_Role_level_metrics_df = new_Role_level_metrics_df[['model',
 'Role','genbit_score',
 'percentage_of_female_gender_definition_words',
 'percentage_of_male_gender_definition_words',
 'percentage_of_non_binary_gender_definition_words',
 'percentage_of_trans_gender_definition_words',
 'percentage_of_cis_gender_definition_words']]

new_word_level_metrics_df = new_word_level_metrics_df[[
 'model','Role','word','frequency',
 'female_count',
 'male_count',
 'non_binary_count',
 'trans_count',
 'cis_count',
 'bias_ratio',
 'bias_conditional_ratio',
 'non_binary_bias_ratio',
 'non_binary_bias_conditional_ratio',
 'cis_bias_ratio',
 'cis_bias_conditional_ratio',
 'female_conditional_prob',
 'male_conditional_prob',
 'binary_conditional_prob',
 'non_binary_conditional_prob',
 'trans_conditional_prob',
 'cis_conditional_prob']]

KeyError: "None of [Index(['model', 'Role', 'word', 'frequency', 'female_count', 'male_count',\n       'non_binary_count', 'trans_count', 'cis_count', 'bias_ratio',\n       'bias_conditional_ratio', 'non_binary_bias_ratio',\n       'non_binary_bias_conditional_ratio', 'cis_bias_ratio',\n       'cis_bias_conditional_ratio', 'female_conditional_prob',\n       'male_conditional_prob', 'binary_conditional_prob',\n       'non_binary_conditional_prob', 'trans_conditional_prob',\n       'cis_conditional_prob'],\n      dtype='object')] are in the [columns]"

In [ ]:
# Export metrics to csv
new_Role_level_metrics_df.to_csv("data/genbit_metrics/Role_level_metrics_v5.csv")
new_word_level_metrics_df.to_csv("data/genbit_metrics/word_level_metrics_v5.csv")